# XTTS-v2 Darija Marocain — Demo Colab

Fine-tuning et génération vocale en Darija marocain

**GitHub** : https://github.com/chaimaehde/xtts-darija

**HuggingFace Space** : https://huggingface.co/spaces/chaimaehde/tts-darija-marocain

In [ ]:
# ÉTAPE 1 — Installation des dépendances
!pip install -q coqui-tts soundfile gradio datasets faster-whisper jiwer
!apt-get install -y ffmpeg -q

In [ ]:
# ÉTAPE 2 — Cloner le repo GitHub
!git clone https://github.com/chaimaehde/xtts-darija.git
%cd xtts-darija
import sys
sys.path.append('/content/xtts-darija')

In [ ]:
# ÉTAPE 3 — Préparer les données DODa (30 min)
from data.prepare_dataset import prepare_all
prepare_all(n_samples=650, output_dir='/content/doda_darija')

In [ ]:
# ÉTAPE 4 — Finetuner XTTS-v2 (nécessite GPU T4)
# Activer GPU : Modifier → Paramètres du notebook → GPU T4
from training.finetune import finetune

finetune(
    data_path   = '/content/doda_darija',
    model_dir   = '/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/',
    output_path = '/content/outputs',
    language    = 'ar',
    epochs      = 5,
    batch_size  = 2,
    lr          = 5e-6,
    grad_accum  = 126,
)

In [ ]:
# ÉTAPE 5 — Charger le modèle finetuné
from inference.generate import load_model

model, config = load_model(
    checkpoint_path = '/content/outputs/best_model.pth',
    config_path     = '/content/outputs/config.json',
    vocab_path      = '/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/vocab.json',
)

In [ ]:
# ÉTAPE 6 — Générer un audio en Darija
from inference.generate import generate
from IPython.display import Audio

generate(
    model       = model,
    config      = config,
    text        = 'مرحبا، كيف داير؟ واش كلشي مزيان؟',
    speaker_wav = '/content/doda_darija/wavs/utt_0000.wav',
    output_path = '/content/test_output.wav',
)

Audio('/content/test_output.wav')

In [ ]:
# ÉTAPE 7 — Évaluation WER et CER (optionnel)
from evaluation.evaluate import evaluate_wer_cer
import os

# Générer plusieurs audios de test
phrases_test = [
    'مرحبا، كيف داير؟',
    'واش نتا مزيان؟',
    'الله يحفظك، بارك الله فيك',
]

from inference.generate import generate_batch
audio_files = generate_batch(
    model       = model,
    config      = config,
    texts       = phrases_test,
    speaker_wav = '/content/doda_darija/wavs/utt_0000.wav',
    output_dir  = '/content/eval_audios',
)

results, avg_wer, avg_cer = evaluate_wer_cer(audio_files, phrases_test)

In [ ]:
# ÉTAPE 8 — Lancer l'interface Gradio
from interface.gradio_app import launch_interface
launch_interface(model, config, share=True)
# Un lien public sera généré automatiquement